# Análise Exploratória de Dados (EDA) - Microdados Censo 2022 (RJ)
**Autores:** Ysis Barbosa Gomes & Ygor Barbosa Gomes  
**Docente / Orientador:** Prof. Alexandre Neves Louzada  
**Curso:** Bacharelado em Sistemas de Informação — 6º Período  
**Disciplina:** Linguagem de Programação (Análise e Visualização de Dados com Python)  
**Princípios FAIR:** Reprodutibilidade, Catalogação e Persistência Aberta  
**Fonte dos Dados:** IBGE — Censo Demográfico 2022: Amostra (UF 33 - Rio de Janeiro)

---
### Perguntas de Pesquisa Investigadas
1. **RQ1:** Quais as disparidades no acesso ao saneamento básico e inclusão digital entre áreas urbanas e rurais no RJ e em função da renda domiciliar per capita?
2. **RQ2:** Como a interseccionalidade de gênero, raça e tipologia do núcleo familiar modula o rendimento per capita e a extrema pobreza?
3. **RQ3:** Qual o padrão temporal e etário da mortalidade domiciliar entre 2019 e 2022, destacando a sobremortalidade no ano de ápice da pandemia de COVID-19 (2021)?

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)
print('Ambiente analítico configurado com sucesso.')

## 1. Carregamento dos Dados Processados (Apache Parquet)
Lemos os arquivos processados persistidos na pasta `../data/processed/`.

In [2]:
data_dir = os.path.join('..', 'data', 'processed')
df_dom = pd.read_parquet(os.path.join(data_dir, 'domicilios_rj_clean.parquet'))
df_fam = pd.read_parquet(os.path.join(data_dir, 'familias_rj_clean.parquet'))
df_mort = pd.read_parquet(os.path.join(data_dir, 'mortalidade_rj_clean.parquet'))
df_pess = pd.read_parquet(os.path.join(data_dir, 'pessoas_amostra_rj_clean.parquet'))

print('Domicílios:', df_dom.shape)
print('Famílias:  ', df_fam.shape)
print('Mortalidade:', df_mort.shape)
print('Pessoas:    ', df_pess.shape)

## 2. RQ1: Saneamento Básico e Conectividade Digital por Situação e Renda
Calculamos as taxas de adequação sanitária e conectividade à internet.

In [3]:
resumo_rq1 = df_dom.groupby('situacao_domicilio', observed=False).agg(
    total_domicilios=('id_domicilio', 'count'),
    taxa_esgoto_adequado=('esgotamento_adequado', lambda s: s.mean() * 100),
    taxa_agua_rede=('agua_rede_geral', lambda s: s.mean() * 100),
    taxa_lixo_coletado=('lixo_coletado_adequado', lambda s: s.mean() * 100),
    taxa_saneamento_pleno=('saneamento_basico_pleno', lambda s: s.mean() * 100),
    taxa_internet=('possui_internet', lambda s: s.mean() * 100)
)
resumo_rq1

## 3. RQ2: Chefia Familiar, Gênero, Raça e Renda per Capita
Analisamos a renda mediana por estrato de gênero e raça do responsável.

In [4]:
df_fam_valid = df_fam[
    df_fam['sexo_responsavel'].isin(['Masculino', 'Feminino']) &
    df_fam['raca_responsavel'].isin(['Branca', 'Preta', 'Parda']) &
    (df_fam['rendimento_familiar_per_capita'] >= 0)
]
resumo_rq2 = df_fam_valid.groupby(['raca_responsavel', 'sexo_responsavel']).agg(
    renda_mediana=('rendimento_familiar_per_capita', 'median'),
    taxa_extrema_pobreza=('em_extrema_pobreza', lambda s: s.mean() * 100),
    total_familias=('id_domicilio', 'count')
)
resumo_rq2

## 4. RQ3: Dinâmica Temporal e Demográfica dos Óbitos (2019-2022)
Avaliamos o aumento expressivo dos óbitos em 2021 durante a pandemia de COVID-19.

In [5]:
obitos_ano = df_mort.groupby('ano_obito').agg(
    total_obitos=('id_ordem_falecido', 'count'),
    homens=('cod_sexo_falecido', lambda s: (s == 1).sum()),
    mulheres=('cod_sexo_falecido', lambda s: (s == 2).sum())
).reset_index()
obitos_ano['variacao_vs_2019_pct'] = ((obitos_ano['total_obitos'] - obitos_ano.loc[0, 'total_obitos']) / obitos_ano.loc[0, 'total_obitos']) * 100
obitos_ano

## 5. Conclusões e Síntese
Os experimentos exploratórios comprovam as fortes clivagens regionais e de renda no Estado do Rio de Janeiro, fundamentando a publicação científica web no formato HTML (`docs/index.html`).